In [ ]:
!pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
import os

print("Files currently uploaded to Colab:")

for file_name in os.listdir("/content"):
    if file_name.endswith(".csv"):
        print(file_name)

Files currently uploaded to Colab:
cm1.csv
kc2.csv
jm1.csv
pc1.csv
kc1.csv


In [ ]:
# Load all datasets
datasets = {
    "KC1": pd.read_csv("/content/kc1.csv"),
    "KC2": pd.read_csv("/content/kc2.csv"),
    "PC1": pd.read_csv("/content/pc1.csv"),
    "CM1": pd.read_csv("/content/cm1.csv"),
    "JM1": pd.read_csv("/content/jm1.csv")
}

print("Datasets loaded successfully!")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Datasets loaded successfully!
KC1: (2109, 22)
KC2: (522, 22)
PC1: (1109, 22)
CM1: (498, 22)
JM1: (13204, 22)


In [ ]:
# Create a summary table for all datasets

summary = []

for name, df in datasets.items():

    # Identify the target column
    if "defects" in df.columns:
        target = "defects"
    else:
        target = "problems"

    # Count defective and non-defective modules
    class_counts = df[target].value_counts()

    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Features": df.shape[1] - 1,
        "Target Column": target,
        "Defective": class_counts.iloc[1] if len(class_counts) > 1 else 0,
        "Non-Defective": class_counts.iloc[0],
        "Defect %": round(class_counts.iloc[1] / df.shape[0] * 100, 2) if len(class_counts) > 1 else 0
    })

summary_df = pd.DataFrame(summary)

summary_df

,Dataset,Rows,Features,Target Column,Defective,Non-Defective,Defect %
0,KC1,2109,21,defects,326,1783,15.46
1,KC2,522,21,problems,107,415,20.50
2,PC1,1109,21,defects,77,1032,6.94
3,CM1,498,21,defects,49,449,9.84
4,JM1,13204,21,defects,2103,11101,15.93


In [ ]:
# Function to prepare each dataset

def prepare_dataset(df):
    """
    Separates features (X) and target (y) for any PROMISE dataset.
    Converts target labels to binary values (0 and 1).
    """

    # Identify the target column
    if "defects" in df.columns:
        target = "defects"
    else:
        target = "problems"

    # Features
    X = df.drop(columns=[target])

    # Target
    y = df[target]

    # Convert target labels to binary
    if y.dtype == "object":
        y = y.map({"no": 0, "yes": 1})
    else:
        y = y.astype(int)

    return X, y

In [ ]:
# Test the function on all datasets

for name, df in datasets.items():
    X, y = prepare_dataset(df)

    print(f"\n{name}")
    print(f"Features shape : {X.shape}")
    print(f"Target shape   : {y.shape}")
    print("Class counts:")
    print(y.value_counts())


KC1
Features shape : (2109, 21)
Target shape   : (2109,)
Class counts:
defects
0    1783
1     326
Name: count, dtype: int64

KC2
Features shape : (522, 21)
Target shape   : (522,)
Class counts:
problems
0    415
1    107
Name: count, dtype: int64

PC1
Features shape : (1109, 21)
Target shape   : (1109,)
Class counts:
defects
0    1032
1      77
Name: count, dtype: int64

CM1
Features shape : (498, 21)
Target shape   : (498,)
Class counts:
defects
0    449
1     49
Name: count, dtype: int64

JM1
Features shape : (13204, 21)
Target shape   : (13204,)
Class counts:
defects
0    11101
1     2103
Name: count, dtype: int64


In [ ]:
# Stratified 10-Fold Cross Validation

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print(cv)

StratifiedKFold(n_splits=10, random_state=42, shuffle=True)


In [ ]:
# Evaluation metrics

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print(scoring)

{'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}


In [ ]:
# Logistic Regression pipeline for the original data

lr_original = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

print(lr_original)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))])


In [ ]:
# Function to evaluate one model on one dataset

def evaluate_model(dataset_name, X, y, model, model_name, sampling_method):
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Dataset": dataset_name,
        "Model": model_name,
        "Sampling": sampling_method,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy SD": scores["test_accuracy"].std(),
        "Precision Mean": scores["test_precision"].mean(),
        "Precision SD": scores["test_precision"].std(),
        "Recall Mean": scores["test_recall"].mean(),
        "Recall SD": scores["test_recall"].std(),
        "F1 Mean": scores["test_f1"].mean(),
        "F1 SD": scores["test_f1"].std(),
        "AUC Mean": scores["test_roc_auc"].mean(),
        "AUC SD": scores["test_roc_auc"].std()
    }

    return result

In [ ]:
# Run original Logistic Regression on all five datasets

lr_results = []

for name, df in datasets.items():
    X, y = prepare_dataset(df)

    result = evaluate_model(
        dataset_name=name,
        X=X,
        y=y,
        model=lr_original,
        model_name="Logistic Regression",
        sampling_method="Original"
    )

    lr_results.append(result)
    print(f"{name} completed")

print("All original Logistic Regression experiments completed.")

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed
All original Logistic Regression experiments completed.


In [ ]:
lr_original_results_df = pd.DataFrame(lr_results)

lr_original_results_df.round(4)

,Dataset,Model,Sampling,Accuracy Mean,Accuracy SD,Precision Mean,Precision SD,Recall Mean,Recall SD,F1 Mean,F1 SD,AUC Mean,AUC SD
0,KC1,Logistic Regression,Original,0.8563,0.0088,0.6236,0.0964,0.1991,0.0351,0.2986,0.0418,0.8039,0.0357
1,KC2,Logistic Regression,Original,0.8430,0.0375,0.7173,0.1551,0.4045,0.0941,0.5127,0.1106,0.8415,0.0759
2,PC1,Logistic Regression,Original,0.9288,0.0142,0.4500,0.4583,0.1036,0.0997,0.1622,0.1577,0.8459,0.0519
3,CM1,Logistic Regression,Original,0.8896,0.0182,0.1500,0.3202,0.0400,0.0800,0.0619,0.1243,0.7995,0.1021
4,JM1,Logistic Regression,Original,0.8453,0.0016,0.5902,0.0333,0.0937,0.0079,0.1616,0.0124,0.7228,0.0224


In [ ]:
# Save Logistic Regression (Original) results

lr_original_results_df.to_csv(
    "lr_original_results.csv",
    index=False
)

print("Logistic Regression (Original) results saved successfully.")

Logistic Regression (Original) results saved successfully.


In [ ]:
# Master list to store all experiment results

all_results = []

# Add the original Logistic Regression results
all_results.extend(lr_results)

print(f"Current experiments completed: {len(all_results)}")

Current experiments completed: 5


In [ ]:
# Logistic Regression with Random Over Sampling (ROS)

lr_ros = Pipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomOverSampler(random_state=42)),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

In [ ]:
# Run Logistic Regression with ROS

for name, df in datasets.items():

    X, y = prepare_dataset(df)

    result = evaluate_model(
        dataset_name=name,
        X=X,
        y=y,
        model=lr_ros,
        model_name="Logistic Regression",
        sampling_method="ROS"
    )

    all_results.append(result)

    print(f"{name} completed")

print("ROS experiments completed.")

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed
ROS experiments completed.


In [ ]:
results_df = pd.DataFrame(all_results)

results_df.round(4)

,Dataset,Model,Sampling,Accuracy Mean,Accuracy SD,Precision Mean,Precision SD,Recall Mean,Recall SD,F1 Mean,F1 SD,AUC Mean,AUC SD
0,KC1,Logistic Regression,Original,0.8563,0.0088,0.6236,0.0964,0.1991,0.0351,0.2986,0.0418,0.8039,0.0357
1,KC2,Logistic Regression,Original,0.8430,0.0375,0.7173,0.1551,0.4045,0.0941,0.5127,0.1106,0.8415,0.0759
2,PC1,Logistic Regression,Original,0.9288,0.0142,0.4500,0.4583,0.1036,0.0997,0.1622,0.1577,0.8459,0.0519
3,CM1,Logistic Regression,Original,0.8896,0.0182,0.1500,0.3202,0.0400,0.0800,0.0619,0.1243,0.7995,0.1021
4,JM1,Logistic Regression,Original,0.8453,0.0016,0.5902,0.0333,0.0937,0.0079,0.1616,0.0124,0.7228,0.0224
5,KC1,Logistic Regression,ROS,0.7255,0.0311,0.3263,0.0354,0.7145,0.0507,0.4472,0.0389,0.8050,0.0369
6,KC2,Logistic Regression,ROS,0.7742,0.0574,0.4717,0.0808,0.6945,0.1583,0.5564,0.0954,0.8390,0.0757
7,PC1,Logistic Regression,ROS,0.7917,0.0407,0.2192,0.0617,0.7411,0.1240,0.3361,0.0807,0.8474,0.0577
8,CM1,Logistic Regression,ROS,0.7834,0.0537,0.2789,0.0987,0.7150,0.2214,0.3965,0.1230,0.7954,0.1089
9,JM1,Logistic Regression,ROS,0.7161,0.0109,0.3034,0.0156,0.6043,0.0410,0.4039,0.0216,0.7252,0.0224


In [ ]:
# Function to run experiments on all datasets

def run_experiment(model, model_name, sampling_method):

    for name, df in datasets.items():

        X, y = prepare_dataset(df)

        result = evaluate_model(
            dataset_name=name,
            X=X,
            y=y,
            model=model,
            model_name=model_name,
            sampling_method=sampling_method
        )

        all_results.append(result)

        print(f"{name} completed")

    print(f"\n{sampling_method} experiments completed.\n")

In [ ]:
lr_rus = Pipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomUnderSampler(random_state=42)),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

run_experiment(
    lr_rus,
    "Logistic Regression",
    "RUS"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

RUS experiments completed.



In [ ]:
lr_smote = Pipeline([
    ("scaler", StandardScaler()),
    ("sampler", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

run_experiment(
    lr_smote,
    "Logistic Regression",
    "SMOTE"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

SMOTE experiments completed.



In [ ]:
results_df = pd.DataFrame(all_results)

lr_all_results = results_df[
    results_df["Model"] == "Logistic Regression"
].reset_index(drop=True)

lr_all_results.round(4)

,Dataset,Model,Sampling,Accuracy Mean,Accuracy SD,Precision Mean,Precision SD,Recall Mean,Recall SD,F1 Mean,F1 SD,AUC Mean,AUC SD
0,KC1,Logistic Regression,Original,0.8563,0.0088,0.6236,0.0964,0.1991,0.0351,0.2986,0.0418,0.8039,0.0357
1,KC2,Logistic Regression,Original,0.8430,0.0375,0.7173,0.1551,0.4045,0.0941,0.5127,0.1106,0.8415,0.0759
2,PC1,Logistic Regression,Original,0.9288,0.0142,0.4500,0.4583,0.1036,0.0997,0.1622,0.1577,0.8459,0.0519
3,CM1,Logistic Regression,Original,0.8896,0.0182,0.1500,0.3202,0.0400,0.0800,0.0619,0.1243,0.7995,0.1021
4,JM1,Logistic Regression,Original,0.8453,0.0016,0.5902,0.0333,0.0937,0.0079,0.1616,0.0124,0.7228,0.0224
5,KC1,Logistic Regression,ROS,0.7255,0.0311,0.3263,0.0354,0.7145,0.0507,0.4472,0.0389,0.8050,0.0369
6,KC2,Logistic Regression,ROS,0.7742,0.0574,0.4717,0.0808,0.6945,0.1583,0.5564,0.0954,0.8390,0.0757
7,PC1,Logistic Regression,ROS,0.7917,0.0407,0.2192,0.0617,0.7411,0.1240,0.3361,0.0807,0.8474,0.0577
8,CM1,Logistic Regression,ROS,0.7834,0.0537,0.2789,0.0987,0.7150,0.2214,0.3965,0.1230,0.7954,0.1089
9,JM1,Logistic Regression,ROS,0.7161,0.0109,0.3034,0.0156,0.6043,0.0410,0.4039,0.0216,0.7252,0.0224


In [ ]:
lr_all_results.to_csv(
    "logistic_regression_all_results.csv",
    index=False
)

print("All Logistic Regression results saved successfully.")

All Logistic Regression results saved successfully.


In [ ]:
# Random Forest - Original

rf_original = Pipeline([
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

run_experiment(
    rf_original,
    "Random Forest",
    "Original"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

Original experiments completed.



In [ ]:
# Random Forest - ROS

rf_ros = Pipeline([
    ("sampler", RandomOverSampler(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

run_experiment(
    rf_ros,
    "Random Forest",
    "ROS"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

ROS experiments completed.



In [ ]:
# Random Forest - RUS

rf_rus = Pipeline([
    ("sampler", RandomUnderSampler(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

run_experiment(
    rf_rus,
    "Random Forest",
    "RUS"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

RUS experiments completed.



In [ ]:
# Random Forest - SMOTE

rf_smote = Pipeline([
    ("sampler", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

run_experiment(
    rf_smote,
    "Random Forest",
    "SMOTE"
)

KC1 completed
KC2 completed
PC1 completed
CM1 completed
JM1 completed

SMOTE experiments completed.



In [ ]:
results_df = pd.DataFrame(all_results)

results_df.round(4)

,Dataset,Model,Sampling,Accuracy Mean,Accuracy SD,Precision Mean,Precision SD,Recall Mean,Recall SD,F1 Mean,F1 SD,AUC Mean,AUC SD
0,KC1,Logistic Regression,Original,0.8563,0.0088,0.6236,0.0964,0.1991,0.0351,0.2986,0.0418,0.8039,0.0357
1,KC2,Logistic Regression,Original,0.8430,0.0375,0.7173,0.1551,0.4045,0.0941,0.5127,0.1106,0.8415,0.0759
2,PC1,Logistic Regression,Original,0.9288,0.0142,0.4500,0.4583,0.1036,0.0997,0.1622,0.1577,0.8459,0.0519
3,CM1,Logistic Regression,Original,0.8896,0.0182,0.1500,0.3202,0.0400,0.0800,0.0619,0.1243,0.7995,0.1021
4,JM1,Logistic Regression,Original,0.8453,0.0016,0.5902,0.0333,0.0937,0.0079,0.1616,0.0124,0.7228,0.0224
5,KC1,Logistic Regression,ROS,0.7255,0.0311,0.3263,0.0354,0.7145,0.0507,0.4472,0.0389,0.8050,0.0369
6,KC2,Logistic Regression,ROS,0.7742,0.0574,0.4717,0.0808,0.6945,0.1583,0.5564,0.0954,0.8390,0.0757
7,PC1,Logistic Regression,ROS,0.7917,0.0407,0.2192,0.0617,0.7411,0.1240,0.3361,0.0807,0.8474,0.0577
8,CM1,Logistic Regression,ROS,0.7834,0.0537,0.2789,0.0987,0.7150,0.2214,0.3965,0.1230,0.7954,0.1089
9,JM1,Logistic Regression,ROS,0.7161,0.0109,0.3034,0.0156,0.6043,0.0410,0.4039,0.0216,0.7252,0.0224


In [ ]:
results_df.to_csv("Final_Results.csv", index=False)

print("Final results saved successfully!")

Final results saved successfully!


In [ ]:
summary_df.to_csv("Dataset_Summary.csv", index=False)

summary_df

,Dataset,Rows,Features,Target Column,Defective,Non-Defective,Defect %
0,KC1,2109,21,defects,326,1783,15.46
1,KC2,522,21,problems,107,415,20.50
2,PC1,1109,21,defects,77,1032,6.94
3,CM1,498,21,defects,49,449,9.84
4,JM1,13204,21,defects,2103,11101,15.93


In [ ]:
comparison = results_df.pivot_table(
    index=["Dataset", "Model"],
    columns="Sampling",
    values=["Accuracy Mean", "Precision Mean", "Recall Mean", "F1 Mean", "AUC Mean"]
)

comparison.round(4)

AUC Mean                         Accuracy Mean  \
Sampling                    Original     ROS     RUS   SMOTE      Original   
Dataset Model                                                                
CM1     Logistic Regression   0.7995  0.7954  0.8011  0.7901        0.8896   
        Random Forest         0.7315  0.7268  0.7289  0.7232        0.8896   
JM1     Logistic Regression   0.7228  0.7252  0.7244  0.7231        0.8453   
        Random Forest         0.8113  0.8026  0.7604  0.8110        0.8535   
KC1     Logistic Regression   0.8039  0.8050  0.7965  0.8049        0.8563   
        Random Forest         0.8292  0.8055  0.8009  0.8340        0.8620   
KC2     Logistic Regression   0.8415  0.8390  0.8415  0.8403        0.8430   
        Random Forest         0.8127  0.7910  0.8132  0.8091        0.8163   
PC1     Logistic Regression   0.8459  0.8474  0.8306  0.8416        0.9288   
        Random Forest         0.8368  0.8589  0.8279  0.8732        0.9387   

                                                     F1 Mean                  \
Sampling                        ROS     RUS   SMOTE Original     ROS     RUS   
Dataset Model                                                                  
CM1     Logistic Regression  0.7834  0.7353  0.7833   0.0619  0.3965  0.3482   
        Random Forest        0.8695  0.6628  0.8253   0.0286  0.0771  0.2933   
JM1     Logistic Regression  0.7161  0.7155  0.7134   0.1616  0.4039  0.4009   
        Random Forest        0.8424  0.6849  0.8462   0.3193  0.3988  0.4107   
KC1     Logistic Regression  0.7255  0.7259  0.7245   0.2986  0.4472  0.4395   
        Random Forest        0.8412  0.7036  0.8340   0.4161  0.4444  0.4308   
KC2     Logistic Regression  0.7742  0.7933  0.7837   0.5127  0.5564  0.6033   
        Random Forest        0.7837  0.7665  0.7894   0.4986  0.4946  0.5668   
PC1     Logistic Regression  0.7917  0.7809  0.7908   0.1622  0.3361  0.3083   
        Random Forest        0.9297  0.7511  0.9170   0.3741  0.3219  0.2977   

                                    Precision Mean                          \
Sampling                      SMOTE       Original     ROS     RUS   SMOTE   
Dataset Model                                                                
CM1     Logistic Regression  0.3780         0.1500  0.2789  0.2344  0.2641   
        Random Forest        0.0995         0.0500  0.1200  0.1841  0.0861   
JM1     Logistic Regression  0.3994         0.5902  0.3034  0.3017  0.2998   
        Random Forest        0.4467         0.6141  0.5085  0.2925  0.5221   
KC1     Logistic Regression  0.4437         0.6236  0.3263  0.3228  0.3249   
        Random Forest        0.4760         0.6080  0.4922  0.3072  0.4761   
KC2     Logistic Regression  0.5778         0.7173  0.4717  0.4995  0.4834   
        Random Forest        0.5291         0.6068  0.4773  0.4591  0.4862   
PC1     Logistic Regression  0.3332         0.4500  0.2192  0.2004  0.2182   
        Random Forest        0.3707         0.6400  0.5349  0.1859  0.4084   

                            Recall Mean                          
Sampling                       Original     ROS     RUS   SMOTE  
Dataset Model                                                    
CM1     Logistic Regression      0.0400  0.7150  0.7150  0.6750  
        Random Forest            0.0200  0.0600  0.7300  0.1200  
JM1     Logistic Regression      0.0937  0.6043  0.5981  0.5991  
        Random Forest            0.2159  0.3290  0.6899  0.3908  
KC1     Logistic Regression      0.1991  0.7145  0.6928  0.7054  
        Random Forest            0.3247  0.4137  0.7259  0.4872  
KC2     Logistic Regression      0.4045  0.6945  0.7791  0.7309  
        Random Forest            0.4409  0.5236  0.7582  0.5891  
PC1     Logistic Regression      0.1036  0.7411  0.6982  0.7286  
        Random Forest            0.2893  0.2732  0.7643  0.3732

In [ ]:
# Recreate the final results DataFrame
results_df = pd.DataFrame(all_results)

# Save separate CSV files
results_df.to_csv(
    "/content/Final_SDP_Results.csv",
    index=False
)

summary_df.to_csv(
    "/content/Dataset_Summary.csv",
    index=False
)

print("CSV files saved successfully.")

CSV files saved successfully.


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'logistic_regression_all_results.csv', 'cm1.csv', 'kc2.csv', 'jm1.csv', 'Final_Results.csv', 'pc1.csv', 'lr_original_results.csv', 'Dataset_Summary.csv', 'Final_SDP_Results.csv', 'kc1.csv', 'sample_data']


In [ ]:
!pip install openpyxl

In [ ]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir())

Current working directory: /content

Files in current directory:
['.config', 'logistic_regression_all_results.csv', 'cm1.csv', 'kc2.csv', 'jm1.csv', 'Final_Results.csv', 'pc1.csv', 'lr_original_results.csv', 'Dataset_Summary.csv', 'Final_SDP_Results.csv', 'kc1.csv', 'sample_data']


In [ ]:
plot_df = results_df.copy()

plot_df["Sampling"] = pd.Categorical(
    plot_df["Sampling"],
    categories=["Original", "ROS", "RUS", "SMOTE"],
    ordered=True
)

plot_df = plot_df.sort_values(
    by=["Dataset", "Model", "Sampling"]
)

plot_df.head()

,Dataset,Model,Sampling,Accuracy Mean,Accuracy SD,Precision Mean,Precision SD,Recall Mean,Recall SD,F1 Mean,F1 SD,AUC Mean,AUC SD
3,CM1,Logistic Regression,Original,0.889592,0.018235,0.150000,0.320156,0.040,0.080000,0.061905,0.124267,0.799495,0.102117
8,CM1,Logistic Regression,ROS,0.783388,0.053715,0.278922,0.098725,0.715,0.221416,0.396532,0.123039,0.795414,0.108926
13,CM1,Logistic Regression,RUS,0.735265,0.059097,0.234446,0.087107,0.715,0.238799,0.348227,0.113284,0.801141,0.095354
18,CM1,Logistic Regression,SMOTE,0.783347,0.046431,0.264110,0.093174,0.675,0.237960,0.378009,0.130156,0.790091,0.121100
23,CM1,Random Forest,Original,0.889633,0.012910,0.050000,0.150000,0.020,0.060000,0.028571,0.085714,0.731515,0.114566
